<a href="https://colab.research.google.com/github/bittuuu17/Agentic-AI/blob/main/AAI_Week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

####Day 15 ReAct Pattern Deep Dive Think solve act again think

In [ ]:
"""
Day 15: ReAct Pattern (Reasoning + Acting)
Google Colab Version
"""

import json
import requests
from typing import Dict, List, Any, Optional, Tuple
from datetime import datetime, timedelta
import re

print("="*60)
print("DAY 15: ReAct PATTERN AGENT")
print("="*60)

# ========================================
# STEP 1: LOAD API KEY
# ========================================

print("\nLoading API keys...")

try:
    from google.colab import userdata
    PERPLEXITY_API_KEY = userdata.get("PERPLEXITY_API_KEY")
    if PERPLEXITY_API_KEY:
        print("[SUCCESS] Perplexity API key loaded")
    else:
        PERPLEXITY_API_KEY = None
        print("[ERROR] Perplexity API key missing!")
except:
    PERPLEXITY_API_KEY = None
    print("[ERROR] Perplexity API key not found")

print("="*60)

# ========================================
# STEP 2: TOOL FUNCTIONS
# ========================================

def get_weather(city: str, date: Optional[str] = None) -> Dict[str, Any]:
    """Get current or historical weather"""

    print(f"  [TOOL] Fetching weather for {city}" +
          (f" on {date}" if date else " (current)"))

    if date is None or date == datetime.now().strftime("%Y-%m-%d"):
        try:
            url = f"https://wttr.in/{city}?format=j1"
            response = requests.get(url, timeout=5)
            response.raise_for_status()

            data = response.json()
            current = data['current_condition'][0]

            return {
                "city": city,
                "date": "today",
                "temperature": f"{current['temp_C']}°C",
                "temp_c": int(current['temp_C']),
                "condition": current['weatherDesc'][0]['value'],
                "is_historical": False
            }

        except Exception as e:
            return {"error": f"Failed: {str(e)}"}

    else:
        import random
        temp = random.randint(8, 18)
        return {
            "city": city,
            "date": date,
            "temperature": f"{temp}°C",
            "temp_c": temp,
            "condition": "Partly cloudy",
            "is_historical": True
        }


def calculator(expression: str) -> Dict[str, Any]:
    """Calculate math expression"""

    print(f"  [TOOL] Calculating: {expression}")

    try:
        allowed_chars = set("0123456789+-*/(). ")
        if not all(c in allowed_chars for c in expression):
            raise ValueError("Invalid characters")

        result_value = eval(expression)

        return {
            "expression": expression,
            "result": float(result_value)
        }

    except Exception as e:
        return {"error": f"Failed: {str(e)}"}


# ========================================
# STEP 3: ReAct AGENT CLASS
# ========================================

class ReActAgent:
    """
    ReAct Pattern Agent

    Alternates between:
    - Thought: Reasoning about next step
    - Action: Executing a tool
    - Observation: Seeing the result
    """

    def __init__(self, max_iterations: int = 5):
        """
        Initialize ReAct agent

        Parameters:
            max_iterations: Maximum number of thought-action-observation cycles
        """
        self.max_iterations = max_iterations
        self.api_key = PERPLEXITY_API_KEY
        self.tools = {
            "get_weather": get_weather,
            "calculator": calculator,
            "FINISH": None
        }

    def _get_tool_descriptions(self) -> str:
        """Get formatted tool descriptions for prompt"""
        return """Available tools:

1. get_weather
   Purpose: Get weather for a city (current or historical)
   Input: {"city": "CityName", "date": "YYYY-MM-DD" (optional)}
   Example: {"city": "NYC"}

2. calculator
   Purpose: Perform mathematical calculations
   Input: {"expression": "math expression"}
   Example: {"expression": "25 * 4"}

3. FINISH
   Purpose: Return final answer to user
   Input: {"answer": "your final answer"}
   Example: {"answer": "NYC is warmer than London"}
"""

    def _ask_llm_for_next_step(self, query: str, history: List[Dict[str, Any]]) -> Tuple[Optional[str], Optional[str], Optional[Dict]]:
        """
        Ask LLM for next thought, action, and action input

        Returns:
            (thought, action, action_input) or (None, None, None) if failed
        """

        if not self.api_key:
            return None, None, None

        # Build context from history
        context = f"Original query: {query}\n\n"

        if history:
            context += "Previous steps:\n"
            for step in history:
                context += f"\nThought: {step['thought']}\n"
                context += f"Action: {step['action']}\n"
                if step.get('action_input'):
                    context += f"Action Input: {json.dumps(step['action_input'])}\n"
                context += f"Observation: {step['observation']}\n"

        system_prompt = f"""You are a ReAct agent. You solve problems by alternating between Thought, Action, and Observation.

{self._get_tool_descriptions()}

RESPOND IN THIS EXACT FORMAT:

Thought: [Your reasoning about what to do next]
Action: [Tool name: get_weather, calculator, or FINISH]
Action Input: {{"param": "value"}}

CRITICAL RULES:
1. ONE thought, ONE action per response
2. After each action (except FINISH), wait for observation
3. Use FINISH action when you have the final answer
4. When using FINISH, Action Input must have "answer" key

Example for "Compare weather in NYC and London":

Iteration 1:
Thought: I need to get weather for NYC first
Action: get_weather
Action Input: {{"city": "NYC"}}

(System returns observation: "15°C, Clear")

Iteration 2:
Thought: Now I need London weather to compare
Action: get_weather
Action Input: {{"city": "London"}}

(System returns observation: "10°C, Rainy")

Iteration 3:
Thought: NYC (15°C) is warmer than London (10°C) by 5 degrees
Action: FINISH
Action Input: {{"answer": "NYC is warmer than London. NYC is 15°C (Clear) while London is 10°C (Rainy), making NYC 5 degrees warmer."}}

NOW RESPOND TO THE CURRENT SITUATION:
"""

        user_message = context + "\nWhat should I think and do next?"

        try:
            response = requests.post(
                "https://api.perplexity.ai/chat/completions",
                headers={
                    "Authorization": f"Bearer {self.api_key}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": "sonar-pro",
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_message}
                    ],
                    "temperature": 0.0
                },
                timeout=30
            )

            if response.status_code != 200:
                return None, None, None

            data = response.json()
            content = data["choices"][0]["message"]["content"].strip()

            # Parse Thought, Action, Action Input
            thought_match = re.search(r'Thought:\s*(.+?)(?=\nAction:|\n\n|$)', content, re.DOTALL)
            action_match = re.search(r'Action:\s*(\w+)', content)
            action_input_match = re.search(r'Action Input:\s*(\{.+?\})', content, re.DOTALL)

            if not thought_match or not action_match:
                return None, None, None

            thought = thought_match.group(1).strip()
            action = action_match.group(1).strip()

            action_input = None
            if action_input_match:
                try:
                    action_input_text = action_input_match.group(1).strip()
                    action_input_text = action_input_text.replace("```json", "").replace("```", "").strip()
                    action_input = json.loads(action_input_text)
                except json.JSONDecodeError:
                    pass

            return thought, action, action_input

        except Exception as e:
            print(f"[ERROR] LLM call failed: {e}")
            return None, None, None

    def _execute_tool(self, action: str, action_input: Optional[Dict[str, Any]]) -> str:
        """
        Execute a tool and return observation

        Returns:
            Observation string describing what happened
        """

        if action == "FINISH":
            return "Task complete"

        if action not in self.tools:
            return f"Error: Unknown action '{action}'"

        tool_func = self.tools[action]

        try:
            if not action_input:
                return "Error: No action input provided"

            result = tool_func(**action_input)

            if isinstance(result, dict) and "error" in result:
                return f"Error: {result['error']}"

            # Format observation based on tool
            if action == "get_weather":
                return f"Weather in {result['city']}: {result['temperature']}, {result['condition']}"

            elif action == "calculator":
                return f"{result['expression']} = {result['result']}"

            else:
                return str(result)

        except Exception as e:
            return f"Error executing {action}: {str(e)}"

    def process_query(self, query: str, verbose: bool = True) -> Dict[str, Any]:
        """
        Process query using ReAct pattern

        Parameters:
            query: User's question
            verbose: Print detailed trace

        Returns:
            Dict with:
                - query: Original query
                - react_trace: List of steps
                - final_answer: Final answer string
                - total_iterations: Number of iterations
        """

        print(f"\n{'='*60}")
        print(f"USER QUERY: {query}")
        print(f"{'='*60}\n")

        react_trace = []
        iteration = 0
        final_answer = None

        while iteration < self.max_iterations:
            iteration += 1

            if verbose:
                print(f"\n--- ITERATION {iteration} ---")

            # Get next step from LLM
            thought, action, action_input = self._ask_llm_for_next_step(query, react_trace)

            if not thought or not action:
                final_answer = "Error: Could not generate next step"
                if verbose:
                    print(f"[ERROR] Failed to get next step")
                break

            if verbose:
                print(f"Thought: {thought}")
                print(f"Action: {action}")
                print(f"Action Input: {action_input}")

            # Check if finished
            if action == "FINISH":
                if action_input and "answer" in action_input:
                    final_answer = action_input["answer"]
                else:
                    final_answer = thought

                observation = "Task complete"

                if verbose:
                    print(f"Observation: {observation}")

                react_trace.append({
                    "iteration": iteration,
                    "thought": thought,
                    "action": action,
                    "action_input": action_input,
                    "observation": observation
                })

                break

            # Execute action
            observation = self._execute_tool(action, action_input)

            if verbose:
                print(f"Observation: {observation}")

            # Add to trace
            react_trace.append({
                "iteration": iteration,
                "thought": thought,
                "action": action,
                "action_input": action_input,
                "observation": observation
            })

        # If max iterations reached without FINISH
        if final_answer is None:
            final_answer = f"Maximum iterations ({self.max_iterations}) reached without completing task"

        print(f"\n{'='*60}")
        print(f"FINAL ANSWER: {final_answer}")
        print(f"{'='*60}\n")

        return {
            "query": query,
            "react_trace": react_trace,
            "final_answer": final_answer,
            "total_iterations": iteration
        }

    def show_trace(self, result: Dict[str, Any]) -> None:
        """
        Display formatted ReAct trace

        Parameters:
            result: Result from process_query()
        """
        print(f"\n{'='*60}")
        print("ReAct TRACE")
        print(f"{'='*60}")
        print(f"Query: {result['query']}")
        print(f"Total Iterations: {result['total_iterations']}")
        print(f"{'='*60}\n")

        for step in result['react_trace']:
            print(f"Iteration {step['iteration']}:")
            print(f"  Thought: {step['thought']}")
            print(f"  Action: {step['action']}")
            if step['action_input']:
                print(f"  Action Input: {json.dumps(step['action_input'], indent=2)}")
            print(f"  Observation: {step['observation']}")
            print()

        print(f"Final Answer: {result['final_answer']}")
        print(f"{'='*60}\n")


# ========================================
# STEP 4: TEST CASES
# ========================================

def run_tests():
    """Run test cases demonstrating ReAct pattern"""

    if not PERPLEXITY_API_KEY:
        print("[ERROR] Cannot run tests without Perplexity API key")
        return

    agent = ReActAgent(max_iterations=6)

    test_cases = [
        {
            "name": "Test 1: Simple Comparison (2 API calls)",
            "query": "Compare weather in NYC and London, which is warmer?"
        },
        {
            "name": "Test 2: Three Cities (3 API calls + calculation)",
            "query": "What's the average temperature between NYC, London, and Tokyo?"
        },
        {
            "name": "Test 3: Conditional Logic",
            "query": "If it's raining in Paris, tell me weather in London, otherwise tell me weather in NYC"
        }
    ]

    for i, test in enumerate(test_cases, 1):
        print(f"\n\n{'#'*60}")
        print(f"# {test['name']}")
        print(f"{'#'*60}")

        result = agent.process_query(test['query'], verbose=True)

        input("\nPress Enter to continue to next test...")


# ========================================
# STEP 5: INTERACTIVE MODE
# ========================================

def interactive_mode():
    """Interactive ReAct agent"""

    print("\n" + "="*60)
    print("ReAct AGENT - INTERACTIVE MODE")
    print("="*60)
    print("\nThe agent will show its thinking process:")
    print("  Thought  -> What it's reasoning")
    print("  Action   -> What tool it uses")
    print("  Observation -> What it sees")
    print("\nCommands:")
    print("  Type your question | 'test' (run tests) | 'quit'")
    print("="*60 + "\n")

    if not PERPLEXITY_API_KEY:
        print("[ERROR] Perplexity API key required!")
        return

    agent = ReActAgent(max_iterations=6)

    while True:
        try:
            user_input = input("\nYou: ").strip()

            if not user_input:
                continue

            if user_input.lower() == 'quit':
                print("Goodbye!")
                break

            if user_input.lower() == 'test':
                run_tests()
                continue

            # Process query
            result = agent.process_query(user_input, verbose=True)

        except KeyboardInterrupt:
            print("\n\nGoodbye!")
            break
        except Exception as e:
            print(f"\n[ERROR] {e}\n")


# ========================================
# STEP 6: MAIN EXECUTION
# ========================================

if __name__ == "__main__":
    # You can choose:

    # Option 1: Run tests
    run_tests()

    # Option 2: Interactive mode
    #interactive_mode()

####Day 16 Multistep Planning Agent

In [ ]:
"""
Day 16: Multi-Step Planning Agent
Creates plan, executes, revises on failure
"""

import json
import requests
from typing import Dict, List, Any, Optional, Tuple
from datetime import datetime, timedelta
import re
from enum import Enum

print("="*60)
print("DAY 16: MULTI-STEP PLANNING AGENT")
print("="*60)

# ========================================
# STEP 1: LOAD API KEY
# ========================================

print("\nLoading API keys...")

try:
    from google.colab import userdata
    PERPLEXITY_API_KEY = userdata.get("PERPLEXITY_API_KEY")
    if PERPLEXITY_API_KEY:
        print("[SUCCESS] Perplexity API key loaded")
    else:
        PERPLEXITY_API_KEY = None
        print("[ERROR] Perplexity API key missing!")
except:
    PERPLEXITY_API_KEY = None
    print("[ERROR] Perplexity API key not found")

print("="*60)

# ========================================
# STEP 2: STATUS ENUM
# ========================================

class StepStatus(Enum):
    """Status of plan step execution"""
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    SUCCESS = "success"
    FAILED = "failed"
    SKIPPED = "skipped"


# ========================================
# STEP 3: TOOL FUNCTIONS
# ========================================

def get_weather(city: str, date: Optional[str] = None) -> Dict[str, Any]:
    """Get current or historical weather"""

    print(f"  [TOOL] get_weather(city={city}, date={date})")

    if date is None or date == datetime.now().strftime("%Y-%m-%d"):
        try:
            url = f"https://wttr.in/{city}?format=j1"
            response = requests.get(url, timeout=5)
            response.raise_for_status()

            data = response.json()
            current = data['current_condition'][0]

            return {
                "city": city,
                "temperature": f"{current['temp_C']}°C",
                "condition": current['weatherDesc'][0]['value'],
                "status": "success"
            }

        except Exception as e:
            return {"error": f"Failed: {str(e)}", "status": "failed"}

    else:
        import random
        temp = random.randint(15, 25)
        conditions = ["Sunny", "Partly cloudy", "Clear"]
        return {
            "city": city,
            "date": date,
            "temperature": f"{temp}°C",
            "condition": random.choice(conditions),
            "status": "success"
        }


def web_search(query: str) -> Dict[str, Any]:
    """Simulated web search (replace with real API in production)"""

    print(f"  [TOOL] web_search(query={query})")

    # Simulated responses based on query
    query_lower = query.lower()

    if "paris" in query_lower and "attraction" in query_lower:
        return {
            "query": query,
            "results": [
                "Eiffel Tower - Iconic iron tower with city views",
                "Louvre Museum - World's largest art museum",
                "Notre Dame Cathedral - Gothic architecture masterpiece",
                "Arc de Triomphe - Monumental arch at Champs-Élysées",
                "Sacré-Cœur - Basilica at Montmartre hilltop"
            ],
            "status": "success"
        }

    elif "paris" in query_lower and "hotel" in query_lower:
        return {
            "query": query,
            "results": [
                "Hotel Eiffel Turenne - $120/night, near Eiffel Tower",
                "Hotel des Arts Montmartre - $100/night, artistic area",
                "Hotel Le Marais - $130/night, historic district",
                "Ibis Paris - $90/night, budget option",
                "Hotel Saint-Germain - $140/night, central location"
            ],
            "status": "success"
        }

    elif "london" in query_lower and "attraction" in query_lower:
        return {
            "query": query,
            "results": [
                "Big Ben - Iconic clock tower",
                "British Museum - World history and culture",
                "Tower of London - Historic castle and Crown Jewels",
                "London Eye - Giant Ferris wheel",
                "Buckingham Palace - Royal residence"
            ],
            "status": "success"
        }

    else:
        return {
            "query": query,
            "results": ["General search results for: " + query],
            "status": "success"
        }


def calculator(expression: str) -> Dict[str, Any]:
    """Calculate math expression"""

    print(f"  [TOOL] calculator(expression={expression})")

    try:
        allowed_chars = set("0123456789+-*/(). ")
        if not all(c in allowed_chars for c in expression):
            raise ValueError("Invalid characters")

        result_value = eval(expression)

        return {
            "expression": expression,
            "result": float(result_value),
            "status": "success"
        }

    except Exception as e:
        return {
            "error": f"Failed: {str(e)}",
            "status": "failed"
        }


def format_itinerary(data: Dict[str, Any]) -> Dict[str, Any]:
    """Format itinerary from collected data"""

    print(f"  [TOOL] format_itinerary(...)")

    # Simulated formatting
    return {
        "itinerary": "3-day formatted itinerary",
        "days": [
            {
                "day": 1,
                "title": "Arrival & Iconic Sites",
                "activities": data.get("attractions", ["Sightseeing"])[:2]
            },
            {
                "day": 2,
                "title": "Culture & Museums",
                "activities": data.get("attractions", ["Museums"])[2:4]
            },
            {
                "day": 3,
                "title": "Leisure & Departure",
                "activities": ["Shopping", "Local cuisine", "Departure"]
            }
        ],
        "status": "success"
    }


# ========================================
# STEP 4: PLAN STEP CLASS
# ========================================

class PlanStep:
    """
    Represents one step in the plan
    """

    def __init__(self, step_number: int, description: str, tool: str,
                 tool_input: Dict[str, Any], dependencies: List[int] = None):
        """
        Initialize a plan step

        Parameters:
            step_number: Step number (1, 2, 3...)
            description: What this step does
            tool: Tool to use (get_weather, web_search, etc.)
            tool_input: Parameters for the tool
            dependencies: List of step numbers this depends on
        """
        self.step_number = step_number
        self.description = description
        self.tool = tool
        self.tool_input = tool_input
        self.dependencies = dependencies or []
        self.status = StepStatus.PENDING
        self.result = None
        self.error = None
        self.retries = 0
        self.max_retries = 2

    def __str__(self) -> str:
        return f"Step {self.step_number}: {self.description} [{self.status.value}]"

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary"""
        return {
            "step_number": self.step_number,
            "description": self.description,
            "tool": self.tool,
            "tool_input": self.tool_input,
            "status": self.status.value,
            "result": self.result,
            "error": self.error,
            "dependencies": self.dependencies
        }


# ========================================
# STEP 5: PLANNING AGENT CLASS
# ========================================

class PlanningAgent:
    """
    Agent that creates plans and executes them

    Workflow:
    1. Analyze task
    2. Create plan (list of PlanStep objects)
    3. Execute steps sequentially
    4. If step fails, revise plan
    5. Return final result
    """

    def __init__(self):
        """Initialize planning agent"""
        self.api_key = PERPLEXITY_API_KEY
        self.tools = {
            "get_weather": get_weather,
            "web_search": web_search,
            "calculator": calculator,
            "format_itinerary": format_itinerary
        }
        self.current_plan = []
        self.execution_log = []
        self.collected_data = {}

    def _get_tool_descriptions(self) -> str:
        """Get formatted tool descriptions"""
        return """Available tools:

1. get_weather(city, date=None)
   Get weather for a city

2. web_search(query)
   Search the web for information

3. calculator(expression)
   Perform calculations

4. format_itinerary(data)
   Format travel itinerary from collected data
"""

    def _ask_llm_to_create_plan(self, task: str) -> List[PlanStep]:
        """
        Ask LLM to create a plan for the task

        Parameters:
            task: User's task description

        Returns:
            List of PlanStep objects
        """

        if not self.api_key:
            return []

        print(f"\n[PLANNING] Creating plan for: {task}")

        system_prompt = f"""You are a planning agent. Break down tasks into steps.

{self._get_tool_descriptions()}

RESPOND IN THIS EXACT FORMAT:

PLAN:
Step 1: [Description]
Tool: [tool_name]
Input: {{"param": "value"}}

Step 2: [Description]
Tool: [tool_name]
Input: {{"param": "value"}}

...

Example for "Plan a 3-day trip to Paris":

PLAN:
Step 1: Research top Paris attractions
Tool: web_search
Input: {{"query": "Paris top attractions"}}

Step 2: Check Paris weather
Tool: get_weather
Input: {{"city": "Paris"}}

Step 3: Find hotels in Paris
Tool: web_search
Input: {{"query": "Paris budget hotels"}}

Step 4: Calculate estimated budget
Tool: calculator
Input: {{"expression": "120*3 + 300 + 200"}}

Step 5: Create day-by-day itinerary
Tool: format_itinerary
Input: {{"data": "collected"}}

NOW CREATE A PLAN FOR THE GIVEN TASK.
"""

        user_message = f"Task: {task}"

        try:
            response = requests.post(
                "https://api.perplexity.ai/chat/completions",
                headers={
                    "Authorization": f"Bearer {self.api_key}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": "sonar-pro",
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_message}
                    ],
                    "temperature": 0.2
                },
                timeout=30
            )

            if response.status_code != 200:
                print(f"[ERROR] API returned {response.status_code}")
                return []

            data = response.json()
            content = data["choices"][0]["message"]["content"].strip()

            print(f"\n[PLAN RECEIVED]")
            print(content)
            print()

            # Parse plan
            plan_steps = []
            step_pattern = r'Step (\d+):\s*(.+?)\nTool:\s*(\w+)\nInput:\s*(\{.+?\})'

            matches = re.finditer(step_pattern, content, re.DOTALL)

            for match in matches:
                step_num = int(match.group(1))
                description = match.group(2).strip()
                tool = match.group(3).strip()
                input_str = match.group(4).strip()

                try:
                    tool_input = json.loads(input_str)
                except:
                    tool_input = {}

                step = PlanStep(
                    step_number=step_num,
                    description=description,
                    tool=tool,
                    tool_input=tool_input
                )

                plan_steps.append(step)

            return plan_steps

        except Exception as e:
            print(f"[ERROR] Failed to create plan: {e}")
            return []

    def _execute_step(self, step: PlanStep) -> bool:
        """
        Execute a single plan step

        Parameters:
            step: PlanStep to execute

        Returns:
            True if successful, False if failed
        """

        print(f"\n[EXECUTING] {step}")

        step.status = StepStatus.IN_PROGRESS

        # Check if tool exists
        if step.tool not in self.tools:
            step.status = StepStatus.FAILED
            step.error = f"Unknown tool: {step.tool}"
            print(f"[FAILED] {step.error}")
            return False

        # Execute tool
        try:
            tool_func = self.tools[step.tool]
            result = tool_func(**step.tool_input)

            # Check if result indicates failure
            if isinstance(result, dict) and result.get("status") == "failed":
                step.status = StepStatus.FAILED
                step.error = result.get("error", "Unknown error")
                print(f"[FAILED] {step.error}")
                return False

            # Success
            step.status = StepStatus.SUCCESS
            step.result = result

            # Store result in collected data
            step_key = f"step_{step.step_number}"
            self.collected_data[step_key] = result

            # Also store by meaningful name
            if step.tool == "web_search" and "attraction" in step.tool_input.get("query", "").lower():
                self.collected_data["attractions"] = result.get("results", [])
            elif step.tool == "web_search" and "hotel" in step.tool_input.get("query", "").lower():
                self.collected_data["hotels"] = result.get("results", [])
            elif step.tool == "get_weather":
                self.collected_data["weather"] = result
            elif step.tool == "calculator":
                self.collected_data["budget"] = result

            print(f"[SUCCESS] Step completed")
            return True

        except Exception as e:
            step.status = StepStatus.FAILED
            step.error = str(e)
            print(f"[FAILED] {step.error}")
            return False

    def _revise_plan_for_failed_step(self, failed_step: PlanStep) -> bool:
        """
        Revise plan when a step fails

        Parameters:
            failed_step: The step that failed

        Returns:
            True if plan was revised, False if should abort
        """

        print(f"\n[REVISING] Plan due to failure in {failed_step}")

        # Strategy 1: Retry if under max retries
        if failed_step.retries < failed_step.max_retries:
            failed_step.retries += 1
            failed_step.status = StepStatus.PENDING
            print(f"[REVISION] Will retry step (attempt {failed_step.retries + 1}/{failed_step.max_retries + 1})")
            return True

        # Strategy 2: Skip non-critical steps
        if "weather" in failed_step.description.lower():
            failed_step.status = StepStatus.SKIPPED
            print(f"[REVISION] Skipping weather check (non-critical)")
            return True

        # Strategy 3: Use alternative approach
        if failed_step.tool == "web_search":
            # Try with simpler query
            if "query" in failed_step.tool_input:
                original_query = failed_step.tool_input["query"]
                failed_step.tool_input["query"] = original_query.split()[0]  # Use first word only
                failed_step.status = StepStatus.PENDING
                failed_step.retries = 0
                print(f"[REVISION] Simplified search query to: {failed_step.tool_input['query']}")
                return True

        # Strategy 4: Abort
        print(f"[REVISION] Cannot recover from failure, aborting")
        return False

    def execute_plan(self, task: str, verbose: bool = True) -> Dict[str, Any]:
        """
        Main method: Create and execute plan

        Parameters:
            task: User's task description
            verbose: Print detailed logs

        Returns:
            Dict with:
                - task: Original task
                - plan: List of steps
                - execution_log: Execution details
                - final_result: Final output
                - success: Boolean
        """

        print(f"\n{'='*60}")
        print(f"TASK: {task}")
        print(f"{'='*60}")

        # Phase 1: Create Plan
        self.current_plan = self._ask_llm_to_create_plan(task)

        if not self.current_plan:
            return {
                "task": task,
                "plan": [],
                "execution_log": [],
                "final_result": "Failed to create plan",
                "success": False
            }

        print(f"\n[PLAN CREATED] {len(self.current_plan)} steps")
        for step in self.current_plan:
            print(f"  {step}")

        # Phase 2: Execute Plan
        print(f"\n{'='*60}")
        print("EXECUTION PHASE")
        print(f"{'='*60}")

        for step in self.current_plan:
            # Execute step
            success = self._execute_step(step)

            # If failed, try to revise
            if not success:
                can_continue = self._revise_plan_for_failed_step(step)

                if not can_continue:
                    # Abort execution
                    return {
                        "task": task,
                        "plan": [s.to_dict() for s in self.current_plan],
                        "execution_log": self.execution_log,
                        "final_result": f"Execution aborted at step {step.step_number}",
                        "success": False
                    }

                # If revised to retry, execute again
                if step.status == StepStatus.PENDING:
                    success = self._execute_step(step)
                    if not success:
                        # Still failed after retry
                        step.status = StepStatus.SKIPPED

        # Phase 3: Generate Final Result
        print(f"\n{'='*60}")
        print("GENERATING FINAL RESULT")
        print(f"{'='*60}")

        final_result = self._generate_final_result(task)

        return {
            "task": task,
            "plan": [s.to_dict() for s in self.current_plan],
            "collected_data": self.collected_data,
            "final_result": final_result,
            "success": True
        }

    def _generate_final_result(self, task: str) -> str:
        """Generate final result from collected data"""

        # Build result based on collected data
        result_parts = []

        result_parts.append(f"Task: {task}")
        result_parts.append("")

        if "attractions" in self.collected_data:
            result_parts.append("Top Attractions:")
            for i, attraction in enumerate(self.collected_data["attractions"][:5], 1):
                result_parts.append(f"  {i}. {attraction}")
            result_parts.append("")

        if "hotels" in self.collected_data:
            result_parts.append("Recommended Hotels:")
            for i, hotel in enumerate(self.collected_data["hotels"][:3], 1):
                result_parts.append(f"  {i}. {hotel}")
            result_parts.append("")

        if "weather" in self.collected_data:
            weather = self.collected_data["weather"]
            result_parts.append(f"Weather: {weather.get('temperature')}, {weather.get('condition')}")
            result_parts.append("")

        if "budget" in self.collected_data:
            budget = self.collected_data["budget"]
            result_parts.append(f"Estimated Budget: ${budget.get('result', 'N/A')}")
            result_parts.append("")

        result_parts.append("3-Day Itinerary:")
        result_parts.append("  Day 1: Arrival & Iconic Sites")
        result_parts.append("  Day 2: Culture & Museums")
        result_parts.append("  Day 3: Leisure & Departure")

        return "\n".join(result_parts)

    def show_plan_summary(self, result: Dict[str, Any]) -> None:
        """Display formatted plan summary"""

        print(f"\n{'='*60}")
        print("PLAN EXECUTION SUMMARY")
        print(f"{'='*60}")
        print(f"Task: {result['task']}")
        print(f"Success: {result['success']}")
        print(f"{'='*60}\n")

        print("Plan Steps:")
        for step_dict in result['plan']:
            status_symbol = {
                "success": "✓",
                "failed": "✗",
                "skipped": "⊘",
                "pending": "○"
            }.get(step_dict['status'], "?")

            print(f"  {status_symbol} Step {step_dict['step_number']}: {step_dict['description']}")
            print(f"    Tool: {step_dict['tool']}")
            print(f"    Status: {step_dict['status']}")
            if step_dict.get('error'):
                print(f"    Error: {step_dict['error']}")
            print()

        print(f"{'='*60}")
        print("FINAL RESULT")
        print(f"{'='*60}")
        print(result['final_result'])
        print(f"{'='*60}\n")


# ========================================
# STEP 6: TEST CASES
# ========================================

def run_tests():
    """Run test cases"""

    if not PERPLEXITY_API_KEY:
        print("[ERROR] Cannot run tests without API key")
        return

    agent = PlanningAgent()

    test_cases = [
        "Plan a 3-day trip to Paris with budget $2000",
        "Research and compare best hotels in London",
        "Plan a weekend trip to Tokyo"
    ]

    for i, task in enumerate(test_cases, 1):
        print(f"\n\n{'#'*60}")
        print(f"# TEST {i}: {task}")
        print(f"{'#'*60}")

        result = agent.execute_plan(task)
        agent.show_plan_summary(result)

        if i < len(test_cases):
            input("\nPress Enter for next test...")


# ========================================
# STEP 7: INTERACTIVE MODE
# ========================================

def interactive_mode():
    """Interactive planning agent"""

    print("\n" + "="*60)
    print("PLANNING AGENT - INTERACTIVE MODE")
    print("="*60)
    print("\nThe agent will:")
    print("  1. Create a plan for your task")
    print("  2. Execute each step")
    print("  3. Revise if steps fail")
    print("  4. Return final result")
    print("\nCommands:")
    print("  Describe your task | 'test' (run tests) | 'quit'")
    print("="*60 + "\n")

    if not PERPLEXITY_API_KEY:
        print("[ERROR] API key required!")
        return

    agent = PlanningAgent()

    while True:
        try:
            user_input = input("\nYou: ").strip()

            if not user_input:
                continue

            if user_input.lower() == 'quit':
                print("Goodbye!")
                break

            if user_input.lower() == 'test':
                run_tests()
                continue

            # Execute task
            result = agent.execute_plan(user_input)
            agent.show_plan_summary(result)

        except KeyboardInterrupt:
            print("\n\nGoodbye!")
            break
        except Exception as e:
            print(f"\n[ERROR] {e}\n")


# ========================================
# STEP 8: MAIN
# ========================================

if __name__ == "__main__":
    # Choose mode:

    # Option 1: Run tests
    # run_tests()

    # Option 2: Interactive mode
    interactive_mode()


####Day 17 Decomposition of Tasks with dependencies

In [ ]:
"""
Day 17: Goal Decomposition Agent (WITH LLM)
- User types any goal
- LLM breaks it into subtasks + dependencies
- Engine executes subtasks in correct order
- Handles failures gracefully
"""

import json
import requests
from typing import Dict, List, Any, Optional
from enum import Enum


# ========================================
# PART 1: API KEY
# ========================================

print("=" * 60)
print("DAY 17: GOAL DECOMPOSITION AGENT (LLM-POWERED)")
print("=" * 60)

print("\nLoading API keys...")

try:
    from google.colab import userdata
    PERPLEXITY_API_KEY = userdata.get("PERPLEXITY_API_KEY")
    if PERPLEXITY_API_KEY:
        print("[SUCCESS] Perplexity API key loaded")
    else:
        PERPLEXITY_API_KEY = None
        print("[ERROR] Perplexity API key missing!")
except:
    PERPLEXITY_API_KEY = None
    print("[ERROR] Perplexity API key not found")

print("=" * 60)


# ========================================
# PART 2: STATUS ENUM
# ========================================

class Status(Enum):
    PENDING = "pending"
    BLOCKED = "blocked"
    READY = "ready"
    RUNNING = "running"
    DONE = "done"
    FAILED = "failed"


# ========================================
# PART 3: TOOL REGISTRY
# All available tools the agent can use.
# LLM picks which tool each subtask uses.
# ========================================

def tool_search(shared_data: Dict, query: str) -> str:
    """
    Simulated web search tool.
    In production, replace with real search API.
    """
    print(f"    [TOOL] search(query='{query}')")

    query_lower = query.lower()

    # Simulated responses based on keywords
    if "flight" in query_lower:
        shared_data["flights_found"] = True
        shared_data["flight_price"] = 450
        shared_data["flight_dates"] = "Feb 10-13"
        return "Found flights: Air France $450, Emirates $480, Turkish $420"

    elif "hotel" in query_lower:
        shared_data["hotels_found"] = True
        shared_data["hotel_price_per_night"] = 100
        return "Found hotels: Hotel Marais $100/night, Hotel Eiffel $120/night, Ibis $80/night"

    elif "restaurant" in query_lower or "food" in query_lower:
        shared_data["restaurants_found"] = True
        return "Found restaurants: Le Petit Bistro, Cafe de Flore, Chez Pierre"

    elif "attraction" in query_lower or "things to do" in query_lower:
        shared_data["attractions_found"] = True
        return "Top attractions: Eiffel Tower, Louvre Museum, Notre Dame, Arc de Triomphe"

    elif "recipe" in query_lower or "ingredient" in query_lower:
        shared_data["recipe_found"] = True
        return "Recipe found: Requires pasta, bacon, eggs, parmesan, black pepper"

    elif "weather" in query_lower:
        shared_data["weather_checked"] = True
        return "Weather: 18 degrees Celsius, Sunny, perfect for sightseeing"

    else:
        return f"Search results for '{query}': General information found"


def tool_calculate(shared_data: Dict, expression: str) -> str:
    """
    Calculator tool. Safely evaluates math expressions.
    """
    print(f"    [TOOL] calculate(expression='{expression}')")

    try:
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return f"Error: Invalid characters in '{expression}'"

        result = eval(expression)
        shared_data["last_calculation"] = result
        return f"{expression} = {result}"

    except Exception as e:
        return f"Calculation error: {e}"


def tool_book(shared_data: Dict, what: str) -> str:
    """
    Simulated booking tool (flights, hotels, restaurants).
    """
    print(f"    [TOOL] book(what='{what}')")

    what_lower = what.lower()

    if "flight" in what_lower:
        if not shared_data.get("flights_found"):
            raise Exception("Cannot book flight - search for flights first")
        shared_data["flight_booked"] = True
        price = shared_data.get("flight_price", 450)
        shared_data["total_spent"] = shared_data.get("total_spent", 0) + price
        return f"Flight booked successfully. Cost: ${price}"

    elif "hotel" in what_lower:
        if not shared_data.get("hotels_found"):
            raise Exception("Cannot book hotel - search for hotels first")
        shared_data["hotel_booked"] = True
        nights = 3
        price_per_night = shared_data.get("hotel_price_per_night", 100)
        total = price_per_night * nights
        shared_data["total_spent"] = shared_data.get("total_spent", 0) + total
        return f"Hotel booked: {nights} nights at ${price_per_night}/night = ${total}"

    elif "restaurant" in what_lower:
        if not shared_data.get("restaurants_found"):
            raise Exception("Cannot book restaurant - search for restaurants first")
        shared_data["restaurant_booked"] = True
        return "Restaurant reservation confirmed: Le Petit Bistro, 7:30 PM"

    else:
        return f"Booked: {what}"


def tool_create_plan(shared_data: Dict, title: str) -> str:
    """
    Creates a final summary/plan from all collected data.
    """
    print(f"    [TOOL] create_plan(title='{title}')")

    parts = [f"=== {title} ==="]

    if shared_data.get("flight_booked"):
        parts.append("  Flight: Booked (Air France)")
    if shared_data.get("hotel_booked"):
        parts.append("  Hotel: Booked (Hotel Marais, 3 nights)")
    if shared_data.get("restaurant_booked"):
        parts.append("  Restaurant: Reserved (Le Petit Bistro)")
    if shared_data.get("attractions_found"):
        parts.append("  Attractions: Eiffel Tower, Louvre, Notre Dame")
    if shared_data.get("weather_checked"):
        parts.append("  Weather: 18C, Sunny")
    if shared_data.get("total_spent"):
        parts.append(f"  Total spent: ${shared_data['total_spent']}")

    parts.append("  Status: Ready to go!")
    return "\n".join(parts)


# Maps tool name (string from LLM) to actual function
TOOL_REGISTRY = {
    "search":      tool_search,
    "calculate":   tool_calculate,
    "book":        tool_book,
    "create_plan": tool_create_plan,
}


# ========================================
# PART 4: SUBTASK CLASS
# ========================================

class SubTask:
    """
    One subtask in the plan.
    LLM generates: id, name, description, tool, tool_input, dependencies
    Engine adds: status, result, error at runtime
    """

    def __init__(self, id: int, name: str, description: str,
                 tool: str, tool_input: Dict[str, Any],
                 dependencies: List[int]):
        self.id = id
        self.name = name
        self.description = description
        self.tool = tool                  # Which tool to call (e.g. "search")
        self.tool_input = tool_input      # Parameters for the tool
        self.dependencies = dependencies  # Subtask IDs that must finish first
        self.status = Status.PENDING
        self.result = None
        self.error = None

    def __str__(self) -> str:
        return f"[{self.status.value.upper():>7}] Task {self.id}: {self.name}"


# ========================================
# PART 5: GOAL DECOMPOSITION ENGINE
# ========================================

class GoalDecompositionEngine:
    """
    1. Receives subtasks (from LLM)
    2. Calculates execution order (topological sort)
    3. Executes each subtask using the correct tool
    4. Blocks downstream tasks if upstream fails
    """

    def __init__(self):
        self.subtasks: Dict[int, SubTask] = {}
        self.execution_order: List[int] = []
        self.shared_data: Dict[str, Any] = {}

    def add_subtasks(self, subtasks: List[SubTask]) -> None:
        for task in subtasks:
            self.subtasks[task.id] = task

    def _calculate_execution_order(self) -> List[int]:
        """
        Topological sort: resolve dependencies recursively.
        A task only appears AFTER all its dependencies.
        """
        ordered = []
        visited = set()

        def resolve(task_id: int) -> None:
            if task_id in visited:
                return
            task = self.subtasks[task_id]
            for dep_id in task.dependencies:
                resolve(dep_id)
            visited.add(task_id)
            ordered.append(task_id)

        for task_id in self.subtasks:
            resolve(task_id)

        return ordered

    def _update_statuses(self) -> None:
        """Update PENDING tasks to BLOCKED or READY based on dependencies"""
        for task in self.subtasks.values():
            if task.status in (Status.DONE, Status.RUNNING, Status.FAILED):
                continue

            if not task.dependencies:
                task.status = Status.READY
            else:
                all_deps_done = all(
                    self.subtasks[dep_id].status == Status.DONE
                    for dep_id in task.dependencies
                )
                task.status = Status.READY if all_deps_done else Status.BLOCKED

    def _execute_subtask(self, task: SubTask) -> bool:
        """
        Execute one subtask by calling its tool with its input.
        """
        print(f"\n  [RUNNING] Task {task.id}: {task.name}")
        print(f"            {task.description}")

        task.status = Status.RUNNING

        # Look up the tool function
        tool_func = TOOL_REGISTRY.get(task.tool)
        if not tool_func:
            task.status = Status.FAILED
            task.error = f"Unknown tool: '{task.tool}'"
            print(f"  [FAILED]  Task {task.id}: unknown tool '{task.tool}'")
            return False

        try:
            # Call tool: shared_data is always first arg, then tool_input params
            result = tool_func(self.shared_data, **task.tool_input)

            task.status = Status.DONE
            task.result = result
            self.shared_data[f"task_{task.id}"] = result

            print(f"  [DONE]    Task {task.id}: {task.name}")
            print(f"            Result: {result}")
            return True

        except Exception as e:
            task.status = Status.FAILED
            task.error = str(e)
            print(f"  [FAILED]  Task {task.id}: {task.name}")
            print(f"            Error: {e}")
            return False

    def execute(self, goal_name: str) -> Dict[str, Any]:
        """
        Main execution loop.
        """
        print(f"\n{'=' * 60}")
        print(f"GOAL: {goal_name}")
        print(f"{'=' * 60}")

        # Calculate order
        self.execution_order = self._calculate_execution_order()

        # Show plan
        print(f"\n[PLAN] {len(self.subtasks)} subtasks, execution order: {self.execution_order}")
        print(f"\nDependency Map:")
        for task_id in self.execution_order:
            task = self.subtasks[task_id]
            deps = task.dependencies if task.dependencies else ["none"]
            print(f"  Task {task.id}: {task.name}")
            print(f"    Tool: {task.tool}({task.tool_input})")
            print(f"    Depends on: {deps}")

        # Initial status update
        self._update_statuses()

        # Execute
        print(f"\n{'=' * 60}")
        print("EXECUTING")
        print(f"{'=' * 60}")

        failed_tasks = []

        for task_id in self.execution_order:
            task = self.subtasks[task_id]

            # If blocked (dependency failed), skip
            if task.status == Status.BLOCKED:
                print(f"\n  [BLOCKED] Task {task.id}: {task.name}")
                print(f"            Skipped - dependency not completed")
                task.status = Status.FAILED
                task.error = "Blocked: dependency failed"
                failed_tasks.append(task_id)
                continue

            # Execute
            success = self._execute_subtask(task)
            if not success:
                failed_tasks.append(task_id)

            # Update statuses after each task
            self._update_statuses()

        # Summary
        print(f"\n{'=' * 60}")
        print("SUMMARY")
        print(f"{'=' * 60}")
        for task_id in self.execution_order:
            print(f"  {self.subtasks[task_id]}")

        success = len(failed_tasks) == 0
        print(f"\n  Result: {'SUCCESS' if success else 'PARTIAL FAILURE'}")
        if failed_tasks:
            print(f"  Failed: {failed_tasks}")

        return {
            "goal": goal_name,
            "success": success,
            "failed_tasks": failed_tasks,
            "shared_data": self.shared_data
        }


# ========================================
# PART 6: LLM PLAN GENERATOR
# Asks Perplexity to decompose user goal
# ========================================

class LLMPlanGenerator:
    """
    Sends user goal to Perplexity.
    Gets back a structured JSON plan.
    Converts JSON into SubTask objects.
    """

    def __init__(self):
        self.api_key = PERPLEXITY_API_KEY

    def _get_system_prompt(self) -> str:
        """
        Tells the LLM exactly what format to respond in.
        """
        return """You are a goal decomposition agent.
The user gives you a goal. You break it into subtasks with dependencies.

AVAILABLE TOOLS (you must only use these):
1. search    - Input: {"query": "what to search for"}
2. calculate - Input: {"expression": "math like 100*3+50"}
3. book      - Input: {"what": "what to book, e.g. flight, hotel, restaurant"}
4. create_plan - Input: {"title": "Final Plan Title"}

RESPOND ONLY with valid JSON. No text before or after. No markdown. No backticks.

FORMAT:
{
  "subtasks": [
    {
      "id": 1,
      "name": "Short name",
      "description": "What this step does",
      "tool": "search",
      "tool_input": {"query": "flights to Paris"},
      "dependencies": []
    },
    {
      "id": 2,
      "name": "Another step",
      "description": "What this does",
      "tool": "book",
      "tool_input": {"what": "flight"},
      "dependencies": [1]
    }
  ]
}

RULES:
- dependencies: [] means no dependencies (runs first)
- dependencies: [1, 3] means task must wait for tasks 1 AND 3 to finish
- Always start with search steps (no dependencies)
- book steps must depend on the corresponding search step
- create_plan should be the LAST task, depending on all key tasks
- Use calculate if any budget/cost math is needed
- Keep it between 4-8 subtasks total
- The last task should always be create_plan to summarize everything

EXAMPLE for "Plan a trip to London":
{
  "subtasks": [
    {"id": 1, "name": "Search Flights", "description": "Find flights to London", "tool": "search", "tool_input": {"query": "flights to London"}, "dependencies": []},
    {"id": 2, "name": "Search Hotels", "description": "Find hotels in London", "tool": "search", "tool_input": {"query": "hotels in London"}, "dependencies": []},
    {"id": 3, "name": "Book Flight", "description": "Book the cheapest flight", "tool": "book", "tool_input": {"what": "flight"}, "dependencies": [1]},
    {"id": 4, "name": "Book Hotel", "description": "Book a hotel", "tool": "book", "tool_input": {"what": "hotel"}, "dependencies": [2]},
    {"id": 5, "name": "Calculate Budget", "description": "Total cost", "tool": "calculate", "tool_input": {"expression": "450 + 300"}, "dependencies": [3, 4]},
    {"id": 6, "name": "Create Trip Plan", "description": "Final summary", "tool": "create_plan", "tool_input": {"title": "London Trip Plan"}, "dependencies": [3, 4, 5]}
  ]
}
"""

    def generate_plan(self, goal: str) -> List[SubTask]:
        """
        Sends goal to LLM, parses JSON response, returns SubTask list.

        Parameters:
            goal: User's goal string

        Returns:
            List of SubTask objects, or empty list if failed
        """
        if not self.api_key:
            print("[ERROR] No API key available")
            return []

        print(f"\n[LLM] Generating plan for: '{goal}'")

        try:
            response = requests.post(
                "https://api.perplexity.ai/chat/completions",
                headers={
                    "Authorization": f"Bearer {self.api_key}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": "sonar-pro",
                    "messages": [
                        {"role": "system", "content": self._get_system_prompt()},
                        {"role": "user", "content": f"Goal: {goal}"}
                    ],
                    "temperature": 0.2
                },
                timeout=30
            )

            if response.status_code != 200:
                print(f"[ERROR] API returned status {response.status_code}")
                return []

            data = response.json()
            content = data["choices"][0]["message"]["content"].strip()

            print(f"\n[LLM] Raw response:\n{content}\n")

            # Clean response - remove markdown fences if present
            content = content.replace("```json", "").replace("```", "").strip()

            # Parse JSON
            plan_json = json.loads(content)

            # Convert JSON to SubTask objects
            subtasks = []
            for item in plan_json["subtasks"]:
                subtask = SubTask(
                    id=item["id"],
                    name=item["name"],
                    description=item["description"],
                    tool=item["tool"],
                    tool_input=item["tool_input"],
                    dependencies=item.get("dependencies", [])
                )
                subtasks.append(subtask)

            print(f"[LLM] Plan created: {len(subtasks)} subtasks")
            return subtasks

        except json.JSONDecodeError as e:
            print(f"[ERROR] Failed to parse LLM response as JSON: {e}")
            return []

        except Exception as e:
            print(f"[ERROR] LLM call failed: {e}")
            return []


# ========================================
# PART 7: MAIN AGENT CLASS
# Ties LLM + Engine together
# ========================================

class GoalDecompositionAgent:
    """
    Main agent class.
    1. Takes user goal
    2. Asks LLM to create plan
    3. Passes plan to engine
    4. Engine executes
    """

    def __init__(self):
        self.llm_planner = LLMPlanGenerator()
        self.engine = GoalDecompositionEngine()

    def run(self, goal: str) -> Dict[str, Any]:
        """
        Full pipeline: goal -> LLM plan -> execute

        Parameters:
            goal: What the user wants to accomplish

        Returns:
            Execution result dict
        """
        # Step 1: LLM generates plan
        subtasks = self.llm_planner.generate_plan(goal)

        if not subtasks:
            print("[ERROR] Could not generate plan. Try rephrasing your goal.")
            return {"goal": goal, "success": False, "error": "Plan generation failed"}

        # Step 2: Engine executes plan
        self.engine = GoalDecompositionEngine()  # Fresh engine each run
        self.engine.add_subtasks(subtasks)
        result = self.engine.execute(goal)

        return result


# ========================================
# PART 8: INTERACTIVE MODE
# ========================================

def interactive_mode():
    """Interactive loop for user input"""

    print(f"\n{'=' * 60}")
    print("DAY 17: GOAL DECOMPOSITION AGENT")
    print("=" * 60)
    print("\nHow it works:")
    print("  1. You type a goal (e.g. 'Plan a trip to Paris')")
    print("  2. LLM breaks it into subtasks with dependencies")
    print("  3. Agent executes subtasks in correct order")
    print("  4. You see the full trace")
    print("\nExample goals to try:")
    print("  - Plan a trip to Paris with budget $2000")
    print("  - Book a weekend trip to London")
    print("  - Plan a trip to Tokyo and find restaurants")
    print("  - Organize a vacation to Rome")
    print("\nType 'quit' to exit")
    print("=" * 60)

    if not PERPLEXITY_API_KEY:
        print("\n[ERROR] Perplexity API key required!")
        return

    agent = GoalDecompositionAgent()

    while True:
        try:
            user_input = input("\nYour goal: ").strip()

            if not user_input:
                continue

            if user_input.lower() in ("quit", "q", "exit"):
                print("Goodbye!")
                break

            result = agent.run(user_input)

        except KeyboardInterrupt:
            print("\n\nGoodbye!")
            break
        except Exception as e:
            print(f"\n[ERROR] {e}")


# ========================================
# PART 9: MAIN
# ========================================

if __name__ == "__main__":
    interactive_mode()

####Day 18 Sequential Agent

In [ ]:
"""
Day 18: Sequential Tool Calls
- Tools must run in a specific order
- Output from one tool becomes input for the next
- Chain data through multiple steps
- Real example: Stock Analyzer (fetch → analyze → report)
"""

import json
import requests
from typing import Dict, List, Any, Optional
from datetime import datetime, timedelta
import re


# ========================================
# PART 1: API KEY
# ========================================

print("=" * 60)
print("DAY 18: SEQUENTIAL TOOL CALLS")
print("=" * 60)

print("\nLoading API keys...")

try:
    from google.colab import userdata
    PERPLEXITY_API_KEY = userdata.get("PERPLEXITY_API_KEY")
    if PERPLEXITY_API_KEY:
        print("[SUCCESS] Perplexity API key loaded")
    else:
        PERPLEXITY_API_KEY = None
        print("[ERROR] Perplexity API key missing!")
except:
    PERPLEXITY_API_KEY = None
    print("[ERROR] Perplexity API key not found")

print("=" * 60)


# ========================================
# PART 2: TOOL FUNCTIONS
# Each tool takes previous step's output
# ========================================

def fetch_stock_data(symbol: str) -> Dict[str, Any]:
    """
    Tool 1: Fetch stock price data (simulated).

    In production, use real API like Alpha Vantage, yfinance, etc.

    Parameters:
        symbol: Stock ticker (e.g. "AAPL", "TSLA")

    Returns:
        Dict with price history
    """
    print(f"\n  [TOOL 1] fetch_stock_data(symbol='{symbol}')")

    # Simulated data - in production, fetch from real API
    # This mimics a 5-day price history
    import random

    base_price = {
        "AAPL": 180.0,
        "TSLA": 250.0,
        "GOOGL": 140.0,
        "MSFT": 380.0,
        "AMZN": 170.0,
    }.get(symbol.upper(), 100.0)

    prices = []
    current_price = base_price

    for i in range(5):
        # Random walk: +/- 2% each day
        change_percent = random.uniform(-0.02, 0.02)
        current_price = current_price * (1 + change_percent)

        date = (datetime.now() - timedelta(days=4-i)).strftime("%Y-%m-%d")
        prices.append({
            "date": date,
            "price": round(current_price, 2)
        })

    result = {
        "symbol": symbol.upper(),
        "prices": prices,
        "current_price": prices[-1]["price"],
        "previous_close": prices[-2]["price"]
    }

    print(f"  [RESULT] Fetched {len(prices)} days of data")
    print(f"           Current: ${result['current_price']}")
    print(f"           Previous: ${result['previous_close']}")

    return result


def analyze_stock_data(stock_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Tool 2: Analyze stock data from Tool 1.

    Takes the output of fetch_stock_data and computes analytics.

    Parameters:
        stock_data: Output from fetch_stock_data
                    Must contain: symbol, prices, current_price, previous_close

    Returns:
        Dict with analysis metrics
    """
    print(f"\n  [TOOL 2] analyze_stock_data(symbol={stock_data['symbol']})")

    prices = stock_data["prices"]
    price_values = [p["price"] for p in prices]

    # Calculate metrics
    avg_price = sum(price_values) / len(price_values)
    min_price = min(price_values)
    max_price = max(price_values)
    volatility = max_price - min_price

    # Day-over-day change
    current = stock_data["current_price"]
    previous = stock_data["previous_close"]
    change = current - previous
    change_percent = (change / previous) * 100

    # Trend
    if price_values[-1] > price_values[0]:
        trend = "UPWARD"
    elif price_values[-1] < price_values[0]:
        trend = "DOWNWARD"
    else:
        trend = "FLAT"

    analysis = {
        "symbol": stock_data["symbol"],
        "metrics": {
            "average_price": round(avg_price, 2),
            "min_price": round(min_price, 2),
            "max_price": round(max_price, 2),
            "volatility": round(volatility, 2),
            "current_price": current,
            "change": round(change, 2),
            "change_percent": round(change_percent, 2),
            "trend": trend
        }
    }

    print(f"  [RESULT] Analysis complete")
    print(f"           Trend: {trend}")
    print(f"           Change: {change:+.2f} ({change_percent:+.2f}%)")
    print(f"           Volatility: ${volatility:.2f}")

    return analysis


def generate_report(analysis: Dict[str, Any]) -> str:
    """
    Tool 3: Generate human-readable report from analysis.

    Takes output from analyze_stock_data and creates formatted text.

    Parameters:
        analysis: Output from analyze_stock_data
                  Must contain: symbol, metrics

    Returns:
        Formatted report string
    """
    print(f"\n  [TOOL 3] generate_report(symbol={analysis['symbol']})")

    symbol = analysis["symbol"]
    m = analysis["metrics"]

    # Build report
    lines = []
    lines.append(f"{'=' * 50}")
    lines.append(f"STOCK ANALYSIS REPORT: {symbol}")
    lines.append(f"{'=' * 50}")
    lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("")
    lines.append("PRICE METRICS:")
    lines.append(f"  Current Price:    ${m['current_price']:.2f}")
    lines.append(f"  Average Price:    ${m['average_price']:.2f}")
    lines.append(f"  Price Range:      ${m['min_price']:.2f} - ${m['max_price']:.2f}")
    lines.append(f"  Volatility:       ${m['volatility']:.2f}")
    lines.append("")
    lines.append("PERFORMANCE:")
    lines.append(f"  Daily Change:     ${m['change']:+.2f}")
    lines.append(f"  Change Percent:   {m['change_percent']:+.2f}%")
    lines.append(f"  5-Day Trend:      {m['trend']}")
    lines.append("")

    # Recommendation based on simple rules
    if m['change_percent'] > 2:
        recommendation = "STRONG BUY - Positive momentum"
    elif m['change_percent'] > 0:
        recommendation = "BUY - Slight upward movement"
    elif m['change_percent'] > -2:
        recommendation = "HOLD - Minimal change"
    else:
        recommendation = "SELL - Declining price"

    lines.append(f"RECOMMENDATION:   {recommendation}")
    lines.append(f"{'=' * 50}")

    report = "\n".join(lines)

    print(f"  [RESULT] Report generated ({len(lines)} lines)")

    return report


# ========================================
# PART 3: SEQUENTIAL TOOL CHAIN
# Executes tools in order, passing data
# ========================================

class SequentialToolChain:
    """
    Executes a sequence of tools where each tool receives
    the output of the previous tool.

    Example:
        Tool 1: fetch_stock_data("AAPL") → stock_data
        Tool 2: analyze_stock_data(stock_data) → analysis
        Tool 3: generate_report(analysis) → report

    Each step's output becomes the next step's input.
    """

    def __init__(self):
        self.execution_log = []

    def execute_chain(self, tools: List[tuple], initial_input: Any) -> Any:
        """
        Execute tools sequentially.

        Parameters:
            tools: List of (tool_function, tool_name) tuples
            initial_input: Input for the first tool

        Returns:
            Output from the last tool
        """
        print(f"\n{'=' * 60}")
        print("SEQUENTIAL TOOL CHAIN")
        print(f"{'=' * 60}")
        print(f"Initial input: {initial_input}")
        print(f"Chain length: {len(tools)} tools")

        current_data = initial_input

        for i, (tool_func, tool_name) in enumerate(tools, 1):
            print(f"\n--- Step {i}/{len(tools)}: {tool_name} ---")

            try:
                # Execute tool with current data
                result = tool_func(current_data)

                # Log execution
                self.execution_log.append({
                    "step": i,
                    "tool": tool_name,
                    "input": str(current_data)[:100],  # Truncate for logging
                    "success": True,
                    "output_type": type(result).__name__
                })

                # Update current_data for next step
                current_data = result

            except Exception as e:
                print(f"  [ERROR] Tool {tool_name} failed: {e}")
                self.execution_log.append({
                    "step": i,
                    "tool": tool_name,
                    "input": str(current_data)[:100],
                    "success": False,
                    "error": str(e)
                })
                raise  # Stop chain on failure

        print(f"\n{'=' * 60}")
        print("CHAIN COMPLETE")
        print(f"{'=' * 60}")

        return current_data

    def show_execution_log(self):
        """Display execution log"""
        print(f"\n{'=' * 60}")
        print("EXECUTION LOG")
        print(f"{'=' * 60}")

        for entry in self.execution_log:
            status = "[SUCCESS]" if entry["success"] else "[FAILED]"
            print(f"{status} Step {entry['step']}: {entry['tool']}")
            if entry["success"]:
                print(f"           Output type: {entry['output_type']}")
            else:
                print(f"           Error: {entry.get('error', 'Unknown')}")


# ========================================
# PART 4: LLM-POWERED SEQUENTIAL AGENT
# LLM decides which tools to chain
# ========================================

class SequentialAgent:
    """
    Agent that uses LLM to:
    1. Understand user's goal
    2. Decide which tools to use
    3. Determine the sequence
    4. Execute the chain
    """

    def __init__(self):
        self.api_key = PERPLEXITY_API_KEY
        self.tools = {
            "fetch_stock_data": fetch_stock_data,
            "analyze_stock_data": analyze_stock_data,
            "generate_report": generate_report
        }

    def _get_tool_descriptions(self) -> str:
        """Get formatted tool list for LLM"""
        return """Available tools (must be called in sequence):

1. fetch_stock_data
   Input: symbol (str) - Stock ticker like "AAPL"
   Output: Dict with price history
   Purpose: Fetches recent stock prices

2. analyze_stock_data
   Input: stock_data (Dict) - Output from fetch_stock_data
   Output: Dict with analysis metrics
   Purpose: Computes statistics and trends

3. generate_report
   Input: analysis (Dict) - Output from analyze_stock_data
   Output: str (formatted report)
   Purpose: Creates human-readable summary

RULES:
- Tools MUST be called in order: fetch → analyze → generate
- Output from one tool becomes input to the next
- Do not skip tools in the chain
"""

    def _ask_llm_for_plan(self, goal: str) -> List[str]:
        """
        Ask LLM which tools to use for the goal.

        Parameters:
            goal: User's request

        Returns:
            List of tool names in order
        """
        if not self.api_key:
            # Fallback: assume full chain
            return ["fetch_stock_data", "analyze_stock_data", "generate_report"]

        system_prompt = f"""{self._get_tool_descriptions()}

USER'S GOAL: Determine which tools are needed and in what order.

RESPOND ONLY with a JSON array of tool names in execution order.
Example: ["fetch_stock_data", "analyze_stock_data", "generate_report"]

Do not include any other text.
"""

        try:
            response = requests.post(
                "https://api.perplexity.ai/chat/completions",
                headers={
                    "Authorization": f"Bearer {self.api_key}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": "sonar-pro",
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": goal}
                    ],
                    "temperature": 0.1
                },
                timeout=30
            )

            if response.status_code != 200:
                print(f"[WARN] LLM API failed, using default chain")
                return ["fetch_stock_data", "analyze_stock_data", "generate_report"]

            data = response.json()
            content = data["choices"][0]["message"]["content"].strip()

            # Clean response
            content = content.replace("```json", "").replace("```", "").strip()

            # Parse JSON array
            tool_sequence = json.loads(content)

            print(f"\n[LLM] Determined tool sequence: {tool_sequence}")
            return tool_sequence

        except Exception as e:
            print(f"[WARN] LLM planning failed ({e}), using default chain")
            return ["fetch_stock_data", "analyze_stock_data", "generate_report"]

    def run(self, goal: str, initial_input: str) -> Any:
        """
        Full pipeline:
        1. LLM determines tool sequence
        2. Execute tools in sequence
        3. Return final result

        Parameters:
            goal: What the user wants (used for planning)
            initial_input: Starting data (e.g. stock symbol)

        Returns:
            Final output from the chain
        """
        print(f"\n{'=' * 60}")
        print(f"SEQUENTIAL AGENT")
        print(f"{'=' * 60}")
        print(f"Goal: {goal}")
        print(f"Initial input: {initial_input}")

        # Step 1: LLM determines sequence
        tool_names = self._ask_llm_for_plan(goal)

        # Step 2: Build tool chain
        tools = []
        for name in tool_names:
            if name not in self.tools:
                print(f"[ERROR] Unknown tool: {name}")
                continue
            tools.append((self.tools[name], name))

        # Step 3: Execute chain
        chain = SequentialToolChain()
        result = chain.execute_chain(tools, initial_input)

        # Step 4: Show log
        chain.show_execution_log()

        return result


# ========================================
# PART 5: PRACTICE EXAMPLES
# Different sequential patterns
# ========================================

def example_1_basic_chain():
    """
    Example 1: Basic 3-step chain without LLM
    Manual sequence: fetch → analyze → report
    """
    print("\n" + "#" * 60)
    print("# EXAMPLE 1: BASIC SEQUENTIAL CHAIN")
    print("#" * 60)

    chain = SequentialToolChain()

    tools = [
        (fetch_stock_data, "fetch_stock_data"),
        (analyze_stock_data, "analyze_stock_data"),
        (generate_report, "generate_report")
    ]

    symbol = "AAPL"
    report = chain.execute_chain(tools, symbol)

    print("\n" + "=" * 60)
    print("FINAL REPORT:")
    print("=" * 60)
    print(report)

    chain.show_execution_log()


def example_2_partial_chain():
    """
    Example 2: Partial chain (only fetch + analyze)
    Stop before report generation
    """
    print("\n" + "#" * 60)
    print("# EXAMPLE 2: PARTIAL CHAIN (no report)")
    print("#" * 60)

    chain = SequentialToolChain()

    tools = [
        (fetch_stock_data, "fetch_stock_data"),
        (analyze_stock_data, "analyze_stock_data")
    ]

    symbol = "TSLA"
    analysis = chain.execute_chain(tools, symbol)

    print("\n" + "=" * 60)
    print("ANALYSIS OUTPUT:")
    print("=" * 60)
    print(json.dumps(analysis, indent=2))

    chain.show_execution_log()


def example_3_llm_agent():
    """
    Example 3: LLM decides the sequence
    """
    if not PERPLEXITY_API_KEY:
        print("\n[SKIP] Example 3 requires API key")
        return

    print("\n" + "#" * 60)
    print("# EXAMPLE 3: LLM-POWERED AGENT")
    print("#" * 60)

    agent = SequentialAgent()

    goal = "Analyze GOOGL stock and give me a detailed report"
    symbol = "GOOGL"

    report = agent.run(goal, symbol)

    print("\n" + "=" * 60)
    print("FINAL REPORT:")
    print("=" * 60)
    print(report)


def example_4_multiple_stocks():
    """
    Example 4: Run chain for multiple stocks
    Demonstrates reusability
    """
    print("\n" + "#" * 60)
    print("# EXAMPLE 4: MULTIPLE STOCKS")
    print("#" * 60)

    symbols = ["AAPL", "TSLA", "MSFT"]

    tools = [
        (fetch_stock_data, "fetch_stock_data"),
        (analyze_stock_data, "analyze_stock_data"),
        (generate_report, "generate_report")
    ]

    for symbol in symbols:
        chain = SequentialToolChain()
        report = chain.execute_chain(tools, symbol)
        print("\n" + report)


# ========================================
# PART 6: INTERACTIVE MODE
# ========================================

def interactive_mode():
    """Interactive sequential agent"""

    print(f"\n{'=' * 60}")
    print("DAY 18: SEQUENTIAL TOOL CALLS")
    print(f"{'=' * 60}")
    print("\nHow it works:")
    print("  1. Enter a stock symbol (e.g. AAPL, TSLA, GOOGL)")
    print("  2. Agent executes: fetch → analyze → generate report")
    print("  3. Each tool's output feeds into the next tool")
    print("  4. You see the final report")
    print("\nCommands:")
    print("  Stock symbol (e.g. 'AAPL') | 'examples' | 'quit'")
    print(f"{'=' * 60}\n")

    agent = SequentialAgent()

    while True:
        try:
            user_input = input("\nStock symbol (or command): ").strip()

            if not user_input:
                continue

            if user_input.lower() in ("quit", "q", "exit"):
                print("Goodbye!")
                break

            if user_input.lower() == "examples":
                print("\n[Running examples...]")
                example_1_basic_chain()

                choice = input("\nRun more examples? (y/n): ").strip().lower()
                if choice == "y":
                    example_2_partial_chain()
                    if PERPLEXITY_API_KEY:
                        example_3_llm_agent()
                continue

            # Treat as stock symbol
            symbol = user_input.upper()

            if PERPLEXITY_API_KEY:
                # Use LLM agent
                goal = f"Analyze {symbol} stock and generate a detailed report"
                report = agent.run(goal, symbol)
            else:
                # Direct chain without LLM
                chain = SequentialToolChain()
                tools = [
                    (fetch_stock_data, "fetch_stock_data"),
                    (analyze_stock_data, "analyze_stock_data"),
                    (generate_report, "generate_report")
                ]
                report = chain.execute_chain(tools, symbol)
                chain.show_execution_log()

            print("\n" + "=" * 60)
            print("FINAL REPORT:")
            print("=" * 60)
            print(report)

        except KeyboardInterrupt:
            print("\n\nGoodbye!")
            break
        except Exception as e:
            print(f"\n[ERROR] {e}")


# ========================================
# PART 7: MAIN
# ========================================

if __name__ == "__main__":
    # Choose mode:

    # Option 1: Run examples
    # example_1_basic_chain()
    # example_2_partial_chain()
    # example_3_llm_agent()  # Requires API key
    # example_4_multiple_stocks()

    # Option 2: Interactive mode
    interactive_mode()